# Lists, Dictionaries, and Loops

In the previous notebook we processed data from a single satellite: one name, one battery reading, one status check. That was enough to learn variables, types, and conditionals, but real analysis rarely involves a single value.

The constellation has 20 satellites, each sending five readings. We need data structures that can hold multiple values at once, and a way to process every value without writing 20 copies of the same code. The explainer article introduced three ways to organise data: ordered sequences, labelled lookups, and unique membership groups. This notebook puts all three into practice.

By the end of this notebook we will be able to:

- Create and manipulate **lists**, **dictionaries**, and **sets**
- Write **for loops** to iterate over collections
- Apply filtering and aggregation logic to answer questions about a dataset
- Load and explore a JSON file containing real constellation data

---

## 1. Lists: ordered sequences

A **list** is the simplest collection: values in a row, in the order you put them. If we have ever worked with a single column in a spreadsheet, we already know what a list is. We create one with square brackets.

In [ ]:
satellite_names = ["Sentinel-1", "Sentinel-2", "Sentinel-3", "Horizon-1", "Polar-1"]
print(satellite_names)

In [ ]:
battery_readings = [42.4, 60.5, 61.6, 88.3, 75.0]
print(battery_readings)

Lists can hold any type: strings, numbers, booleans, even other lists.

### Indexing

In a spreadsheet, the first row is row 1. In Python, the first item is at index `0`. This is called **zero-based indexing**, and it takes a little getting used to.

In [ ]:
print(satellite_names[0])   # First item
print(satellite_names[2])   # Third item
print(satellite_names[-1])  # Last item (negative indexing counts from the end)

**Negative indexing** is a useful shortcut: `-1` gives us the last item, `-2` the second-to-last. No need to know how long the list is.

### Slicing

A **slice** extracts a portion of a list. The syntax is `list[start:stop]`, where `start` is included and `stop` is excluded.

In [ ]:
print(satellite_names[1:3])   # Items at index 1 and 2
print(satellite_names[:2])    # From the start up to (not including) index 2
print(satellite_names[3:])    # From index 3 to the end

### Length

`len()` returns the number of items in a list.

In [ ]:
print(len(satellite_names))

### Modifying lists

Unlike strings (which cannot be changed once created), lists are **mutable**. We can add, insert, and extend them after creation.

In [ ]:
# Append adds one item to the end
satellite_names.append("Equator-1")
print(satellite_names)

# Extend adds all items from another list
satellite_names.extend(["Equator-2", "Equator-3"])
print(satellite_names)

# Insert adds at a specific position
satellite_names.insert(0, "Command-Alpha")
print(satellite_names)

Here is a practical example. Sentinel-1 has sent five battery readings over the past day. We can use indexing and `len()` to inspect the data:

In [ ]:
sentinel_1_battery = [42.4, 65.7, 76.4, 26.9, 37.3]

print("First reading:", sentinel_1_battery[0])
print("Latest reading:", sentinel_1_battery[-1])
print("Number of readings:", len(sentinel_1_battery))

---

## 2. Dictionaries: labelled data

Lists work well when position is all we need, but satellite telemetry has named fields: an ID, a name, an orbit type, an altitude. Looking up a value by position (`fields[3]`) is fragile. If the field order changes, the code breaks.

A **dictionary** stores data as **key-value pairs**, so we look things up by name instead of position. The explainer compared this to a VLOOKUP table, and the analogy holds. We create one with curly braces.

In [ ]:
satellite = {
    "satellite_id": "SAT-001",
    "name": "Sentinel-1",
    "orbit_type": "LEO",
    "altitude_km": 27901.0
}

print(satellite)

### Accessing values

Use square brackets with the key name to retrieve a value. We can also use `.get()`, which returns `None` (or a default we specify) instead of crashing if the key does not exist.

In [ ]:
print(satellite["name"])
print(satellite["orbit_type"])

In [ ]:
# .get() returns None instead of raising an error if the key doesn't exist
print(satellite.get("fuel_remaining"))       # None — key doesn't exist
print(satellite.get("fuel_remaining", "N/A"))  # You can provide a default value

The difference matters in real code: `satellite["fuel_remaining"]` would crash with a `KeyError`. Using `.get()` is safer when we are not certain the key exists.

### Adding and updating entries

Dictionaries are mutable. We add new entries and update existing ones with the same syntax:

In [ ]:
# Add a new key
satellite["launch_date"] = "2021-01-14"

# Update an existing key
satellite["altitude_km"] = 27950.0

print(satellite)

### Useful dictionary methods

`.keys()`, `.values()`, and `.items()` let us inspect a dictionary's contents. The `.items()` method is especially useful because it gives us both the key and the value together, which is exactly what we need for looping (coming up in section 5).

In [ ]:
print(list(satellite.keys()))    # All key names
print(list(satellite.values()))  # All values

# .items() returns key-value pairs — very useful for looping (we'll see this soon)
for key, value in satellite.items():
    print(f"{key}: {value}")

### Nested dictionaries

Dictionaries can contain other dictionaries (or lists). This is how structured data is typically represented: each satellite record is a dictionary, and its latest reading is a dictionary inside that dictionary.

In [ ]:
reading = {
    "timestamp": "2025-01-28T00:00:00Z",
    "battery_pct": 42.4,
    "signal_strength_dbm": -108.1,
    "temperature_c": 68.4,
    "status": "active"
}

satellite_with_reading = {
    "name": "Sentinel-1",
    "orbit_type": "LEO",
    "latest_reading": reading
}

# Access nested data by chaining square brackets
print(satellite_with_reading["latest_reading"]["battery_pct"])

---

## 3. Sets: unique values

The third collection type handles a problem we have probably solved in spreadsheets with "Remove Duplicates": we have a column of values and we only care about the distinct ones. A **set** is an unordered collection where every value is unique. If we add a duplicate, the set ignores it.

In [ ]:
orbit_types = {"LEO", "MEO", "GEO", "SSO", "LEO", "GEO"}
print(orbit_types)  # Duplicates are removed

In [ ]:
statuses = {"active", "degraded", "active", "active"}
print(statuses)

### Creating a set from a list

This is the most common use: start with a list that has duplicates and extract only the unique values.

In [ ]:
all_ids = ["SAT-001", "SAT-002", "SAT-001", "SAT-003", "SAT-002", "SAT-001"]
unique_ids = set(all_ids)
print(unique_ids)
print(f"{len(all_ids)} entries, {len(unique_ids)} unique")

### Set operations

Sets support operations for comparing groups: **union** (everything in either group), **intersection** (only what appears in both), and **difference** (what is in one but not the other). These replace what would be several VLOOKUP-and-filter steps in a spreadsheet.

In [ ]:
leo_satellites = {"Sentinel-1", "Sentinel-2", "Sentinel-3", "Polar-2"}
low_battery = {"Sentinel-1", "Sentinel-2", "Polar-3", "Polar-5"}

# Union: in either set
print("All mentioned:", leo_satellites | low_battery)

# Intersection: in both sets
print("LEO AND low battery:", leo_satellites & low_battery)

# Difference: in the first set but not the second
print("LEO but not low battery:", leo_satellites - low_battery)

---

## 4. Choosing the right collection

Each collection type solves a different problem. The choice comes down to three questions: Does order matter? Do I need to look things up by name? Do I only care about unique values?

| Collection | Ordered? | Duplicates? | Access by | Best for |
|---|---|---|---|---|
| **List** | Yes | Yes | Index (position) | Sequences where order matters |
| **Dictionary** | Insertion order | Keys must be unique | Key (name) | Labelled data, lookups by name |
| **Set** | No | No | N/A (membership test) | Unique values, group comparisons |

In practice, we will often combine them. A list of dictionaries is how tabular data looks in Python: each dictionary is one row, and the list holds all the rows in order.

---

## 5. For loops: processing collections

We now have collections that hold multiple values. The next question is: how do we do something with every value? In a spreadsheet, we write a formula in one cell and drag it down the column. In Python, we write a **for loop**: one instruction that runs once for each item in the collection.

In [ ]:
names = ["Sentinel-1", "Sentinel-2", "Sentinel-3"]

for name in names:
    print(f"Checking satellite: {name}")

The variable `name` takes on each value in the list, one at a time. The indented block runs for each value. Whether the list has 3 items or 3 million, the code looks the same.

### Looping through dictionary items

In [ ]:
satellite_orbits = {
    "Sentinel-1": "LEO",
    "Horizon-1": "LEO",
    "Polar-1": "MEO",
    "Equator-1": "GEO"
}

for sat_name, orbit in satellite_orbits.items():
    print(f"{sat_name} is in {orbit} orbit")

### The accumulator pattern

A very common pattern in data analysis: start with an initial value (often zero), then update it inside the loop. This is how we calculate sums, counts, and find minimums or maximums. It is the loop equivalent of a running total in a spreadsheet column.

In [ ]:
readings = [42.4, 65.7, 76.4, 26.9, 37.3]

# Sum
total = 0
for value in readings:
    total = total + value
print(f"Total: {total}")
print(f"Average: {total / len(readings):.1f}")

In [ ]:
# Count
low_count = 0
for value in readings:
    if value < 50:
        low_count = low_count + 1
print(f"Readings below 50%: {low_count}")

In [ ]:
# Finding the minimum
lowest = readings[0]
for value in readings:
    if value < lowest:
        lowest = value
print(f"Lowest reading: {lowest}")

Python has built-in shortcuts (`sum()`, `min()`, `max()`) that handle these simple cases. But the accumulator pattern matters because we will need it whenever the logic is more specific than "add everything up". For example: "sum only the readings below 50%" requires a loop with a condition.

### Enumerate: tracking the index

Sometimes we need both the item and its position. **`enumerate()`** gives us both.

In [ ]:
for index, value in enumerate(readings):
    print(f"Reading {index}: {value}%")

---

## 6. Filtering with conditions

Now the loop and the conditional come together. This is the Python version of filtering rows in a spreadsheet: walk through every item, check a condition, and keep only the items that pass.

In [ ]:
battery_levels = [42.4, 65.7, 76.4, 26.9, 37.3, 88.1, 91.5, 15.2]

# Build a filtered list of low battery readings
low_battery_readings = []
for level in battery_levels:
    if level < 50:
        low_battery_readings.append(level)

print(f"Low battery readings: {low_battery_readings}")
print(f"Count: {len(low_battery_readings)} out of {len(battery_levels)}")

In [ ]:
# Filter satellites by status
satellites_data = [
    {"name": "Sentinel-1", "status": "active"},
    {"name": "Horizon-1", "status": "degraded"},
    {"name": "Polar-1", "status": "active"},
    {"name": "Equator-5", "status": "degraded"}
]

degraded = []
for sat in satellites_data:
    if sat["status"] == "degraded":
        degraded.append(sat["name"])

print(f"Degraded satellites: {degraded}")

---

## 7. List comprehensions

The loop-and-append pattern above is so common that Python has a shorthand for it: the **list comprehension**. It does the same work in a single line.

In [ ]:
# The loop version
low_battery_readings = []
for level in battery_levels:
    if level < 50:
        low_battery_readings.append(level)

# The list comprehension version — same result
low_battery_readings = [level for level in battery_levels if level < 50]

print(low_battery_readings)

The general pattern is:

```python
[expression for item in collection if condition]
```

The `if` part is optional. Without it, we get a **transformation**: the expression is applied to every item.

In [ ]:
# Transform: convert all names to uppercase
names = ["Sentinel-1", "Horizon-1", "Polar-1"]
upper_names = [name.upper() for name in names]
print(upper_names)

In [ ]:
# Filter and transform combined
degraded_names = [sat["name"] for sat in satellites_data if sat["status"] == "degraded"]
print(degraded_names)

List comprehensions are widely used in Python and worth getting comfortable with. If the logic gets complicated (nested conditions, multiple steps), use a regular loop instead. Readability matters more than brevity.

---

## 8. Loading and exploring the constellation data

So far we have been working with small lists typed directly into code cells. Now we will load the full constellation dataset from a JSON file: 20 satellites, each with five readings.

**JSON** (JavaScript Object Notation) is a text format for structured data. It maps directly onto the Python collections we have just learned: JSON objects become dictionaries, JSON arrays become lists. Python's built-in `json` module handles the conversion.

In [ ]:
import json

with open("../data/constellation_status.json", "r") as f:
    data = json.load(f)

print(type(data))
print(list(data.keys()))

The `with open(...) as f` pattern opens a file and ensures it gets closed automatically when we are done. `json.load(f)` parses the text into Python dictionaries and lists.

The top-level structure is a dictionary. The `"satellites"` key contains a list of satellite dictionaries, exactly the "list of dictionaries" pattern we discussed in section 4.

In [ ]:
satellites = data["satellites"]
print(f"Constellation: {data['constellation_name']}")
print(f"Operator: {data['operator']}")
print(f"Number of satellites: {len(satellites)}")

In [ ]:
# Look at the first satellite
first = satellites[0]
print(f"ID: {first['satellite_id']}")
print(f"Name: {first['name']}")
print(f"Orbit: {first['orbit_type']}")
print(f"Number of readings: {len(first['readings'])}")

### Navigating the nested structure

Each satellite has a list of readings, and each reading is itself a dictionary. Getting to a specific value means chaining square brackets: first the satellite, then its readings list, then a field within one reading.

In [ ]:
# First reading of the first satellite
first_reading = satellites[0]["readings"][0]
print(first_reading)

In [ ]:
# Battery percentage from the latest reading of the first satellite
latest_battery = satellites[0]["readings"][-1]["battery_pct"]
print(f"{satellites[0]['name']} latest battery: {latest_battery}%")

### Extract all satellite names

A list comprehension makes this a one-liner:

In [ ]:
all_names = [sat["name"] for sat in satellites]
print(all_names)

### Find all orbit types

Using a set comprehension (curly braces instead of square brackets) gives us only the distinct values:

In [ ]:
orbit_types = {sat["orbit_type"] for sat in satellites}  # Set comprehension — note the curly braces
print(f"Orbit types in constellation: {orbit_types}")

### Get the latest reading for each satellite

Combining a loop with dictionary access and f-string formatting, we can build a quick dashboard view of the entire constellation:

In [ ]:
for sat in satellites:
    latest = sat["readings"][-1]
    print(f"{sat['name']:15s} | status: {latest['status']:10s} | battery: {latest['battery_pct']:5.1f}%")

---

## 9. Answering questions with collections and loops

Everything up to this point has been building towards this: using collections, loops, and conditions together to answer real questions about the data. Each question below follows the same approach. Loop through the satellites, apply a condition or accumulator, collect the result.

### Which satellites have a degraded latest reading?

In [ ]:
degraded_satellites = []
for sat in satellites:
    latest = sat["readings"][-1]
    if latest["status"] == "degraded":
        degraded_satellites.append(sat["name"])

print(f"Degraded satellites ({len(degraded_satellites)}): {degraded_satellites}")

### What is the average battery level across the constellation?

Using the accumulator pattern with each satellite's latest reading:

In [ ]:
total_battery = 0
for sat in satellites:
    latest = sat["readings"][-1]
    total_battery += latest["battery_pct"]

avg_battery = total_battery / len(satellites)
print(f"Average battery (latest readings): {avg_battery:.1f}%")

### Which orbit type has the most satellites?

Here the accumulator is a dictionary: we count occurrences by building up a key-value mapping of orbit types to counts.

In [ ]:
orbit_counts = {}
for sat in satellites:
    orbit = sat["orbit_type"]
    if orbit in orbit_counts:
        orbit_counts[orbit] += 1
    else:
        orbit_counts[orbit] = 1

print("Satellites per orbit type:")
for orbit, count in orbit_counts.items():
    print(f"  {orbit}: {count}")

### Which satellite has the weakest signal?

This requires looking at every reading of every satellite, not just the latest. Signal strength is in dBm, where more negative means weaker.

In [ ]:
weakest_name = None
weakest_signal = 0  # dBm values are negative, so 0 is higher than any reading

for sat in satellites:
    for reading in sat["readings"]:
        if reading["signal_strength_dbm"] < weakest_signal:
            weakest_signal = reading["signal_strength_dbm"]
            weakest_name = sat["name"]

print(f"Weakest signal: {weakest_name} at {weakest_signal} dBm")

Notice the **nested loop**: the outer loop walks through each satellite, and the inner loop walks through that satellite's readings. Nested loops are common when working with nested data structures.

---

## 10. Exercises

Use the `satellites` list from the loaded data. Each exercise builds on the techniques covered above.

### Exercise 1: Orbit type breakdown

Extract all unique orbit types from the constellation and count how many satellites are in each type. Print the results.

*Hint: You can use a set to find the unique types, then loop through the satellites to count each one. Or build a dictionary of counts directly (like the orbit_counts example above).*

In [ ]:
# Your code here


### Exercise 2: Low battery alert

Find all satellites where the **latest** reading has a `battery_pct` below 50%. Print each satellite's name and its battery percentage.

*Hint: The latest reading is the last item in the satellite's readings list. Use index `-1`.*

In [ ]:
# Your code here


### Exercise 3: Status summary dictionary

Build a dictionary where each key is a satellite name and each value is that satellite's latest status. The result should look like `{"Sentinel-1": "active", "Sentinel-2": "active", ...}`. Print it.

*Hint: Start with an empty dictionary and add entries inside a loop, or use a dictionary comprehension: `{key_expr: value_expr for item in collection}`.*

In [ ]:
# Your code here


### Exercise 4: Highest average temperature

Find the satellite with the highest average `temperature_c` across all five of its readings. Print the satellite name and its average temperature.

*Hint: For each satellite, sum the temperature values from all its readings and divide by the number of readings. Track the highest average as you go (the accumulator pattern).*

In [ ]:
# Your code here


---

## Summary

We started this notebook with a single satellite and ended with 20, each carrying five readings. Along the way, we learned the three core collection types and the tools for working with them:

- **Lists** store ordered sequences, accessed by index. Satellite names, battery readings, time series data.
- **Dictionaries** store labelled data, accessed by key. A satellite's properties, a single telemetry reading, a count of orbit types.
- **Sets** store unique values and support group comparisons. Distinct orbit types, membership checks.
- **For loops** process every item in a collection. Combined with conditions, they let us filter, count, sum, and find extremes.
- **List comprehensions** condense the filter-and-collect pattern into a single readable line.
- **`json.load()`** reads structured data from a file into the same Python collections we have been using.

The analysis in section 9 worked, but it took quite a lot of code. Each question required its own loop, its own accumulator, its own print statement. In the next notebook, we will learn to **package that logic into functions** and meet **pandas**, a library that replaces most of these loops with one-line operations.